# Production scan: gamma_* = 1 to 20

Builds the BubbleMaster pair-spectrum scan for $\bar\lambda=0.84$, $\gamma_* = 1$--$20$, $kR_* = 1$--$100$ with
`ptbuilder.ic.generate_surrogate`, and writes a SLURM script that runs it on GPU nodes. Change the settings in the
next cell for a different potential or range.

In [ ]:
from pathlib import Path

import ptbuilder as pt
from ptbuilder.ic import generate_surrogate
from ptbuilder.physics import Phi4Potential, PhysicsModel

ROOT = Path(pt.__file__).parent.parent
LAMBDA_BAR = 0.84

model = PhysicsModel(Phi4Potential(LAMBDA_BAR))

GAMMA_STAR_MIN = 1.0
GAMMA_STAR_MAX = 20.0
KR_MIN, KR_MAX = 1.0, 100.0
N_OMEGA = 32   # production values

## Sizing: node count and walltime

Row runtimes span 2-3 orders of magnitude across the grid (seconds at low `gamma_ij` to hours near
`gamma_ij_max = 50`), so choosing the node count is a bin-packing problem. Estimated with largest-row-first scheduling
(each row goes to the least-loaded GPU) and a cost model calibrated on measured runtimes at `gamma_ij = 12, 16, 20, 24`
(fit within ~1-3%), using this scan's own `T_MAX`:

| nodes | slots | makespan | idle GPU-h | utilization |
|---|---|---|---|---|
| 2  | 8  | 23.7h | 0.1  | 100%  |
| 4  | 16 | 11.9h | 0.3  | 99.8% |
| **8**  | **32** | **6.0h**  | **2.7**  | **98.6%** |
| 10 | 40 | 5.9h  | 46.2 | 80.4% |
| 12 | 48 | 5.9h  | 93.4 | 67.0% |

Utilization stays near 99% up to 8 nodes and drops beyond that: the makespan is then set by the single longest row
(~5.9h), so extra nodes only add idle time. **8 nodes (32 GPUs)** is the default. Total estimated work: ~190 GPU-hours.

Rows are dispatched largest-`gamma_ij`-first. Ascending order would take 9.2h on 8 nodes instead of 6.0h for the same
work, because the cheap rows finish early and leave most GPUs idle while the last slow rows run.

The walltime is about twice the 6.0h estimate, since the most expensive rows (`gamma_ij > 24`, ~28% of the work) lie
beyond the calibrated range.

In [ ]:
GPU_ACCOUNT  = 'YOUR_ACCOUNT'   # SLURM account to charge
GPU_NODES    = 8            # see sizing rationale above
GPU_WALLTIME = '12:00:00'   # ~2x the 6.0h makespan estimate

## Build the scan

In [ ]:
scan = generate_surrogate(model, ROOT, f'lb{LAMBDA_BAR:g}', GAMMA_STAR_MIN, GAMMA_STAR_MAX,
                          KR_MIN, KR_MAX, N_OMEGA)
print(f'{len(scan.manifest_rows)} rows, gamma_ij {scan.gamma_ij_grid[0]:.2f}-'
      f'{scan.gamma_ij_grid[-1]:.2f}, T_MAX={scan.t_max:.1f}')

## SLURM script

The SLURM header (constraint, QOS, account) is written for NERSC Perlmutter GPU nodes; adapt it for other clusters.

In [ ]:
def write_slurm_script(scan, gpu_walltime, gpu_nodes, gpu_account, force_overwrite=False,
                       gpu_qos='premium'):
    """Writes scripts/runtime_scan_{tag}_gpu.sh for `scan` (from generate_surrogate).

    Rows are dispatched largest-gamma_ij-first (see the sizing section); row
    numbering and names are unchanged, only the order in which xargs -P picks
    them up."""
    (ROOT / 'logs').mkdir(exist_ok=True)
    gpus_per_node = 4
    n_concurrent  = gpu_nodes * gpus_per_node
    n_tasks       = len(scan.manifest_rows)
    relative_manifest = scan.manifest.relative_to(ROOT)
    relative_output   = scan.output_root.relative_to(ROOT)
    overwrite_flag = ' --overwrite' if force_overwrite else ''

    script_path = ROOT / 'scripts' / f'runtime_scan_{scan.tag}_gpu.sh'
    script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=bm_{scan.tag}_gpu
#SBATCH --output={ROOT}/logs/bm_{scan.tag}_gpu_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos={gpu_qos}
#SBATCH -A {gpu_account}
#SBATCH --nodes={gpu_nodes}
#SBATCH --ntasks={n_concurrent}
#SBATCH --ntasks-per-node={gpus_per_node}
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={gpu_walltime}
set -euo pipefail
REPO_ROOT="{ROOT}"  # fixed at generation time, so sbatch works from any directory
cd "$REPO_ROOT"

run_one() {{
    local row_num=$1
    local line
    line=$(sed -n "$((row_num + 1))p" {relative_manifest})
    IFS=$'\t' read -r index gamma n_t setup name <<< "$line"
    output="{relative_output}/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores bin/bubblemaster_gpu "$setup" "$output/" --param {scan.n_min} --filon-panels-per-osc {scan.panels_per_oscillation} --s-batch-size {scan.s_batch_size}{overwrite_flag}
}}
export -f run_one

seq {n_tasks} -1 1 | xargs -P {n_concurrent} -I ROW bash -c 'run_one "$@"' _ ROW
''')
    script_path.chmod(0o755)
    print(f'Written: {script_path.relative_to(ROOT)}  '
          f'({n_tasks} tasks, {n_concurrent} concurrent slots on {gpu_nodes} nodes)'
          + (', --overwrite' if force_overwrite else ''))
    return script_path


script_path = write_slurm_script(scan, gpu_walltime=GPU_WALLTIME, gpu_nodes=GPU_NODES,
                                 gpu_account=GPU_ACCOUNT)  # gpu_qos defaults to 'premium'
print(f'Submit: sbatch {script_path.relative_to(ROOT)}')